# Spark Modeling
this is the bread and butter of the big data project. Here we shall take the ML pipeline that we built in the 02_modeling.ipynb notebook and rebuild it in pyspark Ml

In [ ]:
# ===============================================
# BASIC SETUP: GIT Auth and mounting google drive
# ================================================

from google.colab import drive
import sys

# Standard mount to access the config file initially
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Group 1/Room-Occupancy-Detection/src')

from utils.config import initialize_project
initialize_project()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Authenticated as: Ande404
Working Directory: /content/drive/.shortcut-targets-by-id/1yYMYQP42V95jrAfCLhvSv1wAR0w7j8iP/Big-data-group-1/Room-Occupancy-Detection


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col

In [4]:
spark = SparkSession.builder.appName("RoomOccupancySparkML").getOrCreate()

schema = StructType([
    StructField("row_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("Light", DoubleType(), True),
    StructField("CO2", DoubleType(), True),
    StructField("HumidityRatio", DoubleType(), True),
    StructField("Occupancy", IntegerType(), True)
])

train_sdf = spark.read.csv("data/raw/datatraining.txt", header=False, schema=schema, sep=",")
test1_sdf = spark.read.csv("data/raw/datatest.txt", header=False, schema=schema, sep=",")
test2_sdf = spark.read.csv("data/raw/datatest2.txt", header=False, schema=schema, sep=",")


In [5]:
train_sdf.printSchema()
train_sdf.show(5)

root
 |-- row_id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Light: double (nullable = true)
 |-- CO2: double (nullable = true)
 |-- HumidityRatio: double (nullable = true)
 |-- Occupancy: integer (nullable = true)

+------+-------------------+-----------+--------+-----+------+-------------------+---------+
|row_id|               date|Temperature|Humidity|Light|   CO2|      HumidityRatio|Occupancy|
+------+-------------------+-----------+--------+-----+------+-------------------+---------+
|  NULL|        Temperature|       NULL|    NULL| NULL|  NULL|               NULL|     NULL|
|     1|2015-02-04 17:51:00|      23.18|  27.272|426.0|721.25|0.00479298817650529|        1|
|     2|2015-02-04 17:51:59|      23.15| 27.2675|429.5| 714.0|0.00478344094931065|        1|
|     3|2015-02-04 17:53:00|      23.15|  27.245|426.0| 713.5|0.00477946352442199|        1|
|     4|2015-02-04 17:54:0

In [6]:
train_sdf = train_sdf.filter(col("row_id").isNotNull()).drop("row_id")
test1_sdf = test1_sdf.filter(col("row_id").isNotNull()).drop("row_id")
test2_sdf = test2_sdf.filter(col("row_id").isNotNull()).drop("row_id")

In [7]:
train_sdf.printSchema()
train_sdf.show(5)

root
 |-- date: string (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Light: double (nullable = true)
 |-- CO2: double (nullable = true)
 |-- HumidityRatio: double (nullable = true)
 |-- Occupancy: integer (nullable = true)

+-------------------+-----------+--------+-----+------+-------------------+---------+
|               date|Temperature|Humidity|Light|   CO2|      HumidityRatio|Occupancy|
+-------------------+-----------+--------+-----+------+-------------------+---------+
|2015-02-04 17:51:00|      23.18|  27.272|426.0|721.25|0.00479298817650529|        1|
|2015-02-04 17:51:59|      23.15| 27.2675|429.5| 714.0|0.00478344094931065|        1|
|2015-02-04 17:53:00|      23.15|  27.245|426.0| 713.5|0.00477946352442199|        1|
|2015-02-04 17:54:00|      23.15|    27.2|426.0|708.25|0.00477150882608175|        1|
|2015-02-04 17:55:00|       23.1|    27.2|426.0| 704.5|0.00475699293331518|        1|
+-------------------+-------

In [8]:
print("Train rows:", train_sdf.count())
print("Test1 rows:", test1_sdf.count())
print("Test2 rows:", test2_sdf.count())

Train rows: 8143
Test1 rows: 2665
Test2 rows: 9752


In [9]:
features = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
label_col = "Occupancy"

In [10]:
# we transform our features into a feature vectors
assembler = VectorAssembler(inputCols=features, outputCol="raw_features")


In [11]:
# Since we are using Logistic Regression model, we have to standazie all features such that
# all the features are on the same scale.
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)


In [15]:
# implementation of the logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=1000, regParam=0.01,elasticNetParam=0.0)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model = lr_pipeline.fit(train_sdf)

lr_pred_test1 = lr_model.transform(test1_sdf)
lr_pred_test2 = lr_model.transform(test2_sdf)

In [16]:
def spark_evaluate(pred_df, label_col="Occupancy", prediction_col="prediction"):
    metrics = {}
    for metric_name in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol=label_col,
            predictionCol=prediction_col,
            metricName=metric_name
        )
        metrics[metric_name] = evaluator.evaluate(pred_df)
    return metrics

print("LR Test1:", spark_evaluate(lr_pred_test1))
print("LR Test2:", spark_evaluate(lr_pred_test2))

LR Test1: {'accuracy': 0.9782363977485928, 'f1': 0.9783501694123646, 'weightedPrecision': 0.9792487447522046, 'weightedRecall': 0.9782363977485928}
LR Test2: {'accuracy': 0.9928219852337982, 'f1': 0.9928535611257423, 'weightedPrecision': 0.9929618455874456, 'weightedRecall': 0.9928219852337983}


# We now have to simulate Scale:
  in this section we simulate scale so that we can demonstrate the powers of spark

In [27]:
# we create a larger dataset by simulating it
big_sdf = train_sdf

for _ in range(150):
    big_sdf = big_sdf.union(train_sdf)

In [28]:
#checking for size
print("Rows:", big_sdf.count())

Rows: 1229593


In [29]:
# we retrain the model and then measure the time it takes to do it
import time

start = time.time()
big_model = lr_pipeline.fit(big_sdf)
end = time.time()

print("Training time:", end - start)

Training time: 266.0899498462677


In [31]:
print("Partitions before:", big_sdf.rdd.getNumPartitions())
spark.sparkContext.defaultParallelism

Partitions before: 151


2